In [ ]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
userInput ="What is Agentic AI? "
model = init_chat_model("groq:llama-3.1-8b-instant")
response = model.invoke(userInput)
response.content



In [13]:
# Step 1 ---Import the necessary modules and classes from langgraph and typing_extensions---
from langgraph.graph import StateGraph,START, END
from langgraph.graph.message import add_messages
from typing_extensions import Annotated
from typing import TypedDict
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition
from langchain_tavily import TavilySearch
from langgraph.checkpoint.memory import InMemorySaver 

memoray = InMemorySaver()

# Step 2 ------------------ State definition for the graph------
class State(TypedDict):
    messages:Annotated[list, add_messages]

# Step 3 -----------------Define the Node function for the graph----------------
def tool_calling_node(state:State):
    return {"messages":[model.invoke(state["messages"])]}

# Step 4 --- Define the tools that will be used in the graph -----
Search_tool = TavilySearch(
    max_results=2,
    topice="general"
)
tools = [Search_tool]
# Step 5 ---------------- Define the graph and add the nodes to it----------------
graph_builder = StateGraph(State)
graph_builder.add_node("tool_calling_node",tool_calling_node)
graph_builder.add_node("tools",ToolNode(tools))
graph_builder.add_edge(START, "tool_calling_node")
graph_builder.add_conditional_edges("tool_calling_node", 
# If there is two nodes , we use add_conditional_edges to route the flow based on the condition
 #If the Lastest message(result) from assistant is a tool call -> tools_condation routes to tools                           
   tools_condition
)
graph_builder.add_edge("tools", "tool_calling_node") 
# After tool execution, we route back to the tool calling node to process 
# the results and generate a response for the user --Agentic AI / React Agent Architecture

# Step 6 ----------------- Execute the graph & Add a memoray ----------------
graph = graph_builder.compile(checkpointer=memoray)

#--Configurable parameter --
config = {"configurable":{"thread_id":"1"}}

#Step 7 ----------------- Test the graph with a query that requires tool usage----------------
User_query = "Hi ,I am Soumik"
response = graph.invoke({
    "messages":User_query
},config=config)

for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Hi ,I am Soumik
================================== Ai Message ==================================

Nice to meet you, Soumik. Is there something I can help you with or would you like to chat?


In [14]:
User_query = "what is my name?"
response = graph.invoke({
    "messages":User_query
},config=config)
#----See the state of the graph after the second query to see the memoray in action ----
for m in response["messages"]:
    m.pretty_print()

================================ Human Message =================================

Hi ,I am Soumik
================================== Ai Message ==================================

Nice to meet you, Soumik. Is there something I can help you with or would you like to chat?
================================ Human Message =================================

what is my name?
================================== Ai Message ==================================

Your name is Soumik.
